# DNA Splice Junction Classification
## K-mer Feature Engineering

In the previous notebooks, I explored the biological background, understood the dataset, cleaned the observations, and examined DNA-specific patterns.

Machine-learning models cannot work directly with DNA strings. Therefore, I need to transform each sequence into a numerical representation.

In this notebook, I will use k-mer feature engineering to represent local nucleotide patterns as numerical features.

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/splice.data")

cleaned_data = pd.read_csv(
    DATA_PATH,
    header=None,
    skipinitialspace=True
)

cleaned_data.columns = ["class", "instance_name", "sequence"]

cleaned_data = cleaned_data.drop_duplicates()

print("Dataset shape:", cleaned_data.shape)

Dataset shape: (3178, 3)


## 2. Choosing the k-mer Size

### WHY

The value of `k` determines how many consecutive nucleotides are included in each k-mer.

For example, with `k = 3`:

`ATGCGA`

produces:

`ATG`, `TGC`, `GCG`, `CGA`

I will use **3-mers (`k = 3`)** for this project for three main reasons:

1. **Captures local sequence patterns**  
   Three nucleotides provide enough context to represent short local patterns that may help distinguish splice-junction classes.

2. **Keeps the feature space manageable**  
   With four standard DNA bases, there can be up to `4³ = 64` possible 3-mers. This provides a relatively compact feature representation compared with larger values of `k`.

3. **Suitable for the SVM and Random Forest models**  
   A compact feature space makes the resulting representation easier to work with while still capturing local nucleotide information.

The dataset also contains ambiguity symbols (`D`, `N`, `R`, and `S`). Therefore, the actual number of possible 3-mers can be larger than 64. However, only the k-mers observed in our dataset will become features.

For these reasons, I will use `k = 3` consistently for the SVM and Random Forest comparison.

In [2]:
def generate_kmers(sequence, k=3):
    return [
        sequence[i:i + k]
        for i in range(len(sequence) - k + 1)
    ]

In [3]:
example_sequence = cleaned_data["sequence"].iloc[0]

generate_kmers(example_sequence, k=3)[:10]

['CCA', 'CAG', 'AGC', 'GCT', 'CTG', 'TGC', 'GCA', 'CAT', 'ATC', 'TCA']

## 3. Creating k-mer Features

### WHY

A machine-learning model needs numerical input.

I will represent each sequence using the frequency of its 3-mers.

Since every sequence contains 60 nucleotides, each sequence produces:

`60 - 3 + 1 = 58`

3-mer positions.

Instead of keeping the positions themselves, I will count how frequently each 3-mer occurs in each sequence.

In [5]:
from collections import Counter

def kmer_counts(sequence, k=3):
    kmers = [
        sequence[i:i + k]
        for i in range(len(sequence) - k + 1)
    ]
    return Counter(kmers)

In [6]:
kmer_features = cleaned_data["sequence"].apply(
    lambda sequence: kmer_counts(sequence, k=3)
)

X = pd.DataFrame(kmer_features.tolist()).fillna(0)

y = cleaned_data["class"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (3178, 97)
Target shape: (3178,)


In [7]:
print("Feature matrix shape:", X.shape)
print("Missing values:", X.isna().sum().sum())
print("Total k-mer counts:", X.sum().sum())

Feature matrix shape: (3178, 97)
Missing values: 0
Total k-mer counts: 184324.0


### FIND

The k-mer transformation produced 97 features for 3,178 sequences.

There are no missing values in the feature matrix because k-mers that do not occur in a sequence are represented by zero.

The feature matrix is now numerical and suitable for machine-learning models.

In [8]:
X.head()

,CCA,CAG,AGC,GCT,CTG,TGC,GCA,CAT,ATC,TCA,...,CTN,TNC,NCG,CCN,CNT,NTT,NGC,GDG,NGA,NNG
0,4.0,5.0,4.0,1.0,3.0,1.0,2.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,1.0,1.0,0.0,1.0,2.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,4.0,3.0,3.0,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3.0,3.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
X.sum().sort_values(ascending=False).head(10)

CTG    5290.0
CAG    4929.0
CCC    4811.0
CCT    4765.0
AGG    4567.0
CTC    4357.0
TGG    4355.0
GGG    4348.0
CCA    4236.0
GAG    4182.0
dtype: float64

## 6. Feature and Target Separation

### DECIDE

I will use the k-mer frequency matrix `X` as the model input and the `class` column `y` as the target.

The `instance_name` column will not be used as a predictive feature because it identifies the original observation rather than representing DNA sequence information.

In [10]:
from pathlib import Path

FEATURE_PATH = Path("../data/processed")
FEATURE_PATH.mkdir(parents=True, exist_ok=True)

In [11]:
X.to_csv(FEATURE_PATH / "kmer_features.csv", index=False)
y.to_csv(FEATURE_PATH / "kmer_target.csv", index=False)

print("K-mer features saved.")
print("Target saved.")

K-mer features saved.
Target saved.


## 7. K-mer Feature Engineering — Summary

### FIND

I transformed each 60-nucleotide DNA sequence into a numerical representation using 3-mer frequency features.

The final feature matrix contains:

- 3,178 observations
- 97 k-mer features
- No missing values

The target contains the three splice-junction classes: `EI`, `IE`, and `N`.

The resulting feature matrix is numerical and ready for model development.

### DECIDE

I will use the 3-mer frequency representation as the input for both the SVM and Random Forest models.

The same feature representation will be used for both models so that their performance can be compared fairly.

Model training and evaluation will be performed in the following notebooks.